In [1]:
import argparse
import os
import pickle

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "2.2.4":
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:

from kawada_env_cnoid import KawadaBaseEnvChoreonoid as RL_Env

In [3]:

# from bex24_env_cnoid import RLEnvChoreonoid as RL_Env

In [4]:
# exp_name = 'kawada-walking-1001'
# ckpt = 1000

In [5]:
exp_name = 'ishiki-walking-randomized'
ckpt = 5000

In [6]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}
env_cfg["rotorInertia"] = 0.3

In [7]:
env = RL_Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        dt=env_cfg['dt'],
        substeps=env_cfg['substeps'],
        show_viewer=True,
    )

In [8]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)


/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [9]:
with torch.no_grad():
        print("cnt :", cnt)   
        actions = policy(obs)
        print("2 : ", actions)
        obs, rews, dones, infos = env.step(actions)
        print(obs)
        torques = env.sim.sbody.getTorques()
        print("torques:", torques)
        cnt += 1

cnt : 0
2 :  tensor([[-0.4922,  1.1619, -0.5262,  0.9142,  0.5129, -2.3008, -0.3137,  1.5030,
          0.7047,  2.9212, -1.3405,  1.0067]], device='cuda:0')


/userdir/samples/../irsl_rl/rl_env_base.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/samples/../irsl_rl/rl_env_cnoid.py:84: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  self.dof_pos = torch.tensor([sbody.angleVector()]).to(torch.float32).to(self.device)


tensor([[-5.5661e-07, -1.2704e-02,  1.4784e-06,  1.8680e-11,  3.6159e-12,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.0591e-08,
          3.0503e-09, -2.0790e-04,  6.8772e-04, -3.7783e-04, -3.3711e-08,
          1.9408e-07, -2.9359e-08, -2.0766e-04,  6.8736e-04, -3.7855e-04,
          4.0624e-08,  7.6493e-07,  7.6095e-08, -5.1980e-03,  1.7193e-02,
         -9.4461e-03, -8.4279e-07,  4.8522e-06, -7.3414e-07, -5.1915e-03,
          1.7185e-02, -9.4641e-03,  1.0156e-06, -4.9217e-01,  1.1619e+00,
         -5.2616e-01,  9.1419e-01,  5.1285e-01, -2.3008e+00, -3.1368e-01,
          1.5030e+00,  7.0475e-01,  2.9212e+00, -1.3405e+00,  1.0067e+00]],
       device='cuda:0')
torques: [ 2.26690835e-06 -2.26026624e-06 -2.76206569e-06  2.83630657e-05
 -6.96788561e-04 -1.30849254e-07  2.26690836e-06 -2.26026624e-06
 -1.83811713e-05  4.86548912e-05 -6.94065067e-04 -1.30849203e-07]


In [10]:
for i in range(50):
    with torch.no_grad():
        actions = policy(obs)
        torques = env.sim.sbody.getTorques()
        print("torquse:",torques)
        obs, rews, dones, infos = env.step(actions)

torquse: [ 2.26690835e-06 -2.26026624e-06 -2.76206569e-06  2.83630657e-05
 -6.96788561e-04 -1.30849254e-07  2.26690836e-06 -2.26026624e-06
 -1.83811713e-05  4.86548912e-05 -6.94065067e-04 -1.30849203e-07]
torquse: [-500.          241.56381552 -157.46970514   30.01843764 -500.
 -500.          500.          113.96720905  373.75854528   20.93125161
 -476.11739352  483.61750876]
torquse: [-500.          500.         -212.69647308 -390.62600497  500.
   30.40570865  500.          500.          500.          500.
  192.91699929 -500.        ]
torquse: [ 500.          -66.2166132  -446.97669738 -500.          500.
  500.         -378.18601413  133.76301489  500.          193.67355334
 -443.03481336 -500.        ]
torquse: [-500.          234.58914382  -17.05116945 -307.23166742 -500.
 -500.          500.           18.48473105  429.3965143    98.4499182
  500.          500.        ]
torquse: [-500.          150.23743419  412.36784419 -500.         -500.
  500.         -500.         -182.860125

In [11]:
# for i in range(500):
#     obs, _ = env.reset()
#     with torch.no_grad():
#         actions = policy(obs)
#         obs, rews, dones, infos = env.step(actions)

In [12]:
env.sim.stop()